## ÉTAPE 7 — ACP (Analyse en Composantes Principales)
Objectif : regrouper les pays selon leur profil agricole


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


## 1. PRÉPARER LES DONNÉES POUR L'ACP


## L'ACP a besoin d'un tableau de la forme :
  - Une ligne  = un pays
  - Une colonne = une culture
  - La valeur  = production totale (1961–2007)


In [ ]:
df = pd.read_csv("JEU_DE_DONNEE.csv")

# Garder uniquement les pays individuels (pas les zones "+")
# et uniquement la production en quantité
df_prod = df[
    (df["element"] == "Production Quantity") &
    (~df["country_or_area"].str.contains(r"\+", na=False))
].dropna(subset=["value"])

# Cultures qu'on va analyser (les plus répandues dans le monde)
cultures_choisies = [
    "cereals_total",
    "roots_and_tubers_total",
    "vegetables_melons_total",
    "fruit_excl_melons_total",
    "oilcrops_primary",
    "pulses_total",
    "wheat",
    "maize",
    "rice_paddy",
    "sugar_cane"
]

# Filtrer sur ces cultures uniquement
df_filtre = df_prod[df_prod["category"].isin(cultures_choisies)]

# Additionner la production totale par pays et par culture (toutes années)
df_agg = (
    df_filtre
    .groupby(["country_or_area", "category"])["value"]
    .sum()
    .reset_index()
)

print(f"Données agrégées : {df_agg.shape[0]} lignes")


## 2. CRÉER LE TABLEAU PAYS × CULTURES (pivot)


In [ ]:
# Chaque pays devient une ligne, chaque culture une colonne
tableau = df_agg.pivot_table(
    index="country_or_area",
    columns="category",
    values="value",
    aggfunc="sum"
)

# Remplacer les valeurs manquantes par 0
# (un pays qui ne produit pas une culture = 0)
tableau = tableau.fillna(0)

print(f"\nTableau pivot : {tableau.shape[0]} pays × {tableau.shape[1]} cultures")
print(tableau.head(5).to_string())


## 3. STANDARDISATION — ÉTAPE OBLIGATOIRE AVANT L'ACP


## Pourquoi ? La Chine produit des milliards de tonnes,
un petit pays produit des milliers. Sans standardisation,
les grands pays domineraient tout l'ACP.
StandardScaler ramène tout à la même échelle (moyenne=0, écart-type=1)


In [ ]:
scaler = StandardScaler()
tableau_scaled = scaler.fit_transform(tableau)

print(f"\nDonnées standardisées : moyenne ≈ {tableau_scaled.mean():.2f}, écart-type ≈ {tableau_scaled.std():.2f}")


## 4. APPLIQUER L'ACP


In [ ]:
# n_components=2 : on compresse toutes les cultures en 2 dimensions
# pour pouvoir tracer un graphique en 2D
pca = PCA(n_components=2)
coordonnees = pca.fit_transform(tableau_scaled)

# Variance expliquée : combien d'information on garde ?
variance_pc1 = pca.explained_variance_ratio_[0] * 100
variance_pc2 = pca.explained_variance_ratio_[1] * 100
variance_totale = variance_pc1 + variance_pc2

print(f"\n--- Variance expliquée ---")
print(f"Composante 1 (PC1) : {variance_pc1:.1f}%")
print(f"Composante 2 (PC2) : {variance_pc2:.1f}%")
print(f"Total conservé     : {variance_totale:.1f}% de l'information")


## 5. CRÉER UN DATAFRAME AVEC LES RÉSULTATS


In [ ]:
df_pca = pd.DataFrame({
    "pays"  : tableau.index,
    "PC1"   : coordonnees[:, 0],
    "PC2"   : coordonnees[:, 1]
})

# Identifier les grands producteurs manuellement
grands_producteurs = [
    "China", "United States of America", "India", "Brazil",
    "France", "Russian Federation", "Australia", "Argentina",
    "Germany", "Canada", "Pakistan", "Indonesia"
]

df_pca["est_grand"] = df_pca["pays"].isin(grands_producteurs)

print(f"\n--- Aperçu des coordonnées ACP ---")
print(df_pca.sort_values("PC1", ascending=False).head(10).to_string(index=False))


## 6. GRAPHIQUE ACP — NUAGE DE POINTS


In [ ]:
fig, ax = plt.subplots(figsize=(14, 9))

# Tous les pays en gris
petits = df_pca[~df_pca["est_grand"]]
ax.scatter(
    petits["PC1"], petits["PC2"],
    color="lightsteelblue", alpha=0.5, s=30, label="Autres pays"
)

# Grands producteurs en bleu foncé
grands = df_pca[df_pca["est_grand"]]
ax.scatter(
    grands["PC1"], grands["PC2"],
    color="#1E3A8A", s=100, zorder=5, label="Grands producteurs"
)

# Annoter les grands producteurs
for _, row in grands.iterrows():
    ax.annotate(
        row["pays"].replace("United States of America", "USA")
                   .replace("Russian Federation", "Russie"),
        xy=(row["PC1"], row["PC2"]),
        xytext=(row["PC1"] + 0.15, row["PC2"] + 0.15),
        fontsize=9, color="#1E3A8A", fontweight="bold"
    )

# Lignes centrales (origine)
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
ax.axvline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)

# Mise en forme
ax.set_title(
    "ACP — Profils agricoles des pays du monde (1961–2007)",
    fontsize=15, fontweight="bold", pad=15
)
ax.set_xlabel(f"Composante 1 — PC1 ({variance_pc1:.1f}% de variance)", fontsize=12)
ax.set_ylabel(f"Composante 2 — PC2 ({variance_pc2:.1f}% de variance)", fontsize=12)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("acp_pays_agricoles.png", dpi=150, bbox_inches="tight")
print("\nGraphique ACP sauvegardé : 'acp_pays_agricoles.png' ✓")
plt.show()


## 7. GRAPHIQUE BONUS — CONTRIBUTION DES CULTURES


## Le "cercle des corrélations" montre quelles cultures
tirent les pays vers droite ou gauche sur le graphique


In [ ]:
composantes = pd.DataFrame(
    pca.components_.T,
    index=cultures_choisies,
    columns=["PC1", "PC2"]
)

fig2, ax2 = plt.subplots(figsize=(10, 8))

# Dessiner les flèches (une par culture)
for culture, row in composantes.iterrows():
    ax2.annotate(
        "", xy=(row["PC1"], row["PC2"]), xytext=(0, 0),
        arrowprops=dict(arrowstyle="->", color="#2563EB", lw=2)
    )
    ax2.text(
        row["PC1"] * 1.1, row["PC2"] * 1.1,
        culture, fontsize=10, ha="center", color="#1E3A8A"
    )

# Cercle de référence
cercle = plt.Circle((0, 0), 1, color="gray", fill=False, linestyle="--", alpha=0.5)
ax2.add_patch(cercle)

ax2.axhline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
ax2.axvline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
ax2.set_xlim(-1.3, 1.3)
ax2.set_ylim(-1.3, 1.3)
ax2.set_title("Cercle des corrélations — Contribution des cultures", fontsize=14, fontweight="bold")
ax2.set_xlabel(f"PC1 ({variance_pc1:.1f}%)", fontsize=12)
ax2.set_ylabel(f"PC2 ({variance_pc2:.1f}%)", fontsize=12)
ax2.set_aspect("equal")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("acp_cercle_correlations.png", dpi=150, bbox_inches="tight")
print("Cercle des corrélations sauvegardé : 'acp_cercle_correlations.png' ✓")
plt.show()


## 8. RÉSUMÉ FINAL


In [ ]:
print("\n=== RÉSUMÉ DE L'ACP ===")
print(f"Pays analysés    : {len(tableau)} pays")
print(f"Cultures         : {len(cultures_choisies)} variables")
print(f"Information gardée : {variance_totale:.1f}%")
print("\nLes pays à droite sur le graphique = grands producteurs céréaliers")
print("Les pays à gauche                  = profils plus diversifiés ou petits")
